In [ ]:
def patch_environment():
    """Patches missing triton.ops, removes invalid BNB_CUDA_VERSION overrides, and handles CPU/GPU fallbacks for Kaggle and Colab runtimes."""
    import os
    import sys
    import types

    # 1. Clean BNB_CUDA_VERSION if it causes missing libcudart.so
    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 2. Patch Triton ops mock for Triton 3+ compatibility
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    m_ops = types.ModuleType("triton.ops")
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 3. Bulletproof fallback for bitsandbytes if native CUDA libs fail or not present
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.44.1"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
# Clone code và cài đặt thư viện cần thiết
!git clone https://github.com/nguyentatmanh/Research-assist.git
%cd Research-assist

!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    datasets==3.0.2 \
    accelerate>=1.1.0 \
    safetensors \
    scikit-learn scipy einops

print("✅ Đã chuẩn bị xong môi trường Colab!")

In [ ]:
# ==============================================================================
# BƯỚC 5: HUẤN LUYỆN AGENT 2 LAYOUTLMv3 (ĐÃ VÁ LỖI PYTHON 3.13 MODULESPEC)
# ==============================================================================
import os, sys, urllib.request

# 1. Tự động tải bản vá train_agent2_t4x2.py mới nhất từ GitHub
dest_path = "training_lab/agent2_reader/train_agent2_t4x2.py"
raw_url = "https://raw.githubusercontent.com/nguyentatmanh/Research-assist/main/training_lab/agent2_reader/train_agent2_t4x2.py"

try:
    req = urllib.request.Request(raw_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(dest_path, 'wb') as out_f:
        out_f.write(resp.read())
    print("✅ Đã cập nhật thành công train_agent2_t4x2.py (Tương thích 100% Python 3.13)!")
except Exception as e:
    print(f"ℹ️ Dùng file local ({e})")

# 2. Khởi chạy huấn luyện Agent 2 Document Reader trên GPU T4 của Colab
!{sys.executable} -u {dest_path} --epochs 3

In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("⏳ Đang chuẩn bị và tối ưu dung lượng...")

# 1. Xóa các checkpoint phụ trung gian (giảm từ 2GB xuống còn đúng ~480MB)
!rm -rf training_lab/agent2_reader/weights/checkpoints

# 2. Dùng lệnh zip của Linux nén siêu tốc (chỉ mất ~5 giây)
!zip -r Agent2_LayoutLMv3_DocLayNet.zip training_lab/agent2_reader/weights/

print("\n🎉 Đã đóng gói thành công: Agent2_LayoutLMv3_DocLayNet.zip (~480MB)")
print("👉 Bấm vào nút màu xanh bên dưới để tải file về máy:")

# 3. Tạo nút bấm tải về 1-click
download_btn = widgets.Button(
    description="📥 Click để tải Agent2_LayoutLMv3_DocLayNet.zip",
    button_style="success",
    layout=widgets.Layout(width="420px", height="50px")
)

def on_click_download(b):
    print("🚀 Trình duyệt đang bắt đầu tải file về máy tính...")
    files.download("Agent2_LayoutLMv3_DocLayNet.zip")

download_btn.on_click(on_click_download)
display(download_btn)

In [ ]:
import os, sys, torch

print("=" * 80)
print("🛡️ [BẢN VÁ 9ab2b4a] HUẤN LUYỆN AGENT 4 FACT-CHECKER (BATCH=4 + GRADIENT CHECKPOINTING)")
print("=" * 80)

# 1. Kéo bản vá mới nhất vừa được tối ưu VRAM về Colab
!git reset --hard origin/main
!git pull origin main

# 2. Xóa sạch bộ nhớ GPU đệm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# 3. Khởi chạy huấn luyện 3 Epochs siêu mượt
!python training_lab/agent4_fact_checker/train_agent4_t4x2.py --epochs 3 --batch-size 4

In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("⏳ Đang dọn dẹp và tối ưu hóa dung lượng Agent 4...")

# 1. Xóa các checkpoint phụ trung gian (chỉ giữ lại model và nhãn xuất bản cuối cùng)
!rm -rf training_lab/agent4_fact_checker/weights/checkpoints
!rm -rf /content/Research-assist/training_lab/agent4_fact_checker/weights/checkpoints

# 2. Dùng lệnh zip Linux nén siêu tốc (~5 giây)
!zip -r Agent4_FactChecker_NLI.zip training_lab/agent4_fact_checker/weights/

print("\n🎉 Đã đóng gói thành công: Agent4_FactChecker_NLI.zip")
print("👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:")

# 3. Tạo nút bấm 1-click
download_btn = widgets.Button(
    description="📥 Click để tải Agent4_FactChecker_NLI.zip",
    button_style="success",
    layout=widgets.Layout(width="420px", height="50px")
)

def on_click_download(b):
    print("🚀 Trình duyệt đang bắt đầu tải Agent 4 về máy tính...")
    files.download("Agent4_FactChecker_NLI.zip")

download_btn.on_click(on_click_download)
display(download_btn)

In [ ]:
import os, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("=" * 80)
print("🎉 ĐÓNG GÓI & TẢI VỀ AGENT 1: ROUTER & PLANNER (EVAL LOSS: 0.0550)")
print("=" * 80)

# 1. Kéo bản vá cập nhật
!git reset --hard origin/main
!git pull origin main

# 2. Dọn dẹp checkpoints phụ trung gian
!rm -rf training_lab/agent1_retrieval/weights/checkpoints
!rm -rf /content/Research-assist/training_lab/agent1_retrieval/weights/checkpoints

# 3. Nén file zip siêu tốc
!zip -r Agent1_Router_Planner.zip training_lab/agent1_retrieval/weights/

print("\n🎉 Đã đóng gói thành công: Agent1_Router_Planner.zip")
print("👉 Bấm nút màu xanh bên dưới để tải về máy tính:")

# 4. Nút bấm 1-click tải về
download_btn = widgets.Button(
    description="📥 Click để tải Agent1_Router_Planner.zip",
    button_style="success",
    layout=widgets.Layout(width="420px", height="50px")
)

def on_click_download(b):
    print("🚀 Đang bắt đầu tải Agent 1 về máy tính...")
    files.download("Agent1_Router_Planner.zip")

download_btn.on_click(on_click_download)
display(download_btn)